In [34]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import csv
import pydeck as pdk
import geopandas as gpd
pd.set_option('display.max_columns', None)

In [35]:
data2017 = pd.read_csv("donnees/élections/leg2017comm.csv", delimiter=",", low_memory=False)
data2017 = data2017[data2017["dep"] == "29"]
data2017["codecommune"] = data2017["codecommune"].astype(int)

data2022 = pd.read_csv("donnees/élections/leg2022comm.csv", delimiter=",", low_memory=False)
data2022 = data2022[data2022["dep"] == "29"]
data2022["codecommune"] = data2022["codecommune"].astype(int)

In [36]:
import geopandas as gpd
import folium

# Charger les données
map_data = gpd.read_file("donnees/communes_2024.geojson")
map_data = map_data[map_data["departement"] == "29"]
map_data["code"] = map_data["code"].astype(int)

# Créer la carte centrée sur le Finistère
center = [map_data.geometry.centroid.y.mean(), map_data.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=9, tiles="OpenStreetMap")

# Ajouter les communes avec popup
for idx, row in map_data.iterrows():
    # Créer un popup avec les informations de la commune
    popup_text = f"<b>{row.get('nom', 'Commune')}</b><br>"

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x: {
            'fillColor': '#3388ff',
            'color': '#ffffff',
            'weight': 1,
            'fillOpacity': 0.5
        },
        highlight_function=lambda x: {
            'fillColor': '#ff7800',
            'color': '#ffffff',
            'weight': 1,
            'fillOpacity': 0.7
        },
        tooltip=row.get('nom', f"Code: {row['departement']}"),
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# Afficher la carte
m.save("carte_finistere.html")
print("Carte interactive sauvegardée dans 'carte_finistere.html'")
m

/var/folders/4z/dn90d96n3v17tkbfc6v_36q40000gn/T/ipykernel_16969/3747715181.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [map_data.geometry.centroid.y.mean(), map_data.geometry.centroid.x.mean()]


Carte interactive sauvegardée dans 'carte_finistere.html'


In [37]:
data_map2017 = map_data.merge(data2017, left_on="code", right_on="codecommune", how="left")
data_map2022 = map_data.merge(data2022, left_on="code", right_on="codecommune", how="left")

In [38]:
import folium
import branca.colormap as cm
import matplotlib.colors as mcolors

# Variable à visualiser
color = 'pvoixFN'  # Attention, en 2017, c'était FN et non pas RN

# Créer la carte centrée sur le Finistère
center = [data_map2017.geometry.centroid.y.mean(), data_map2017.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=9, tiles="OpenStreetMap")

# Obtenir la palette 'autumn' de matplotlib
cmap_mpl = plt.cm.autumn

# Créer une palette de couleurs Folium compatible avec 'autumn'
vmin = data_map2017[color].min() * 100  # Multiplier par 100
vmax = data_map2017[color].max() * 100  # Multiplier par 100

# Extraire les couleurs de la colormap matplotlib 'autumn'
colors_autumn = [mcolors.rgb2hex(cmap_mpl(i)) for i in [0, 0.25, 0.5, 0.75, 1.0]]
colormap = cm.LinearColormap(
    colors=colors_autumn,
    vmin=vmin,
    vmax=vmax,
    caption='Pourcentage de voix FN 2017 (%)'
)

# Ajouter les communes avec coloration selon pvoixFN
for idx, row in data_map2017.iterrows():
    value = row[color]

    # Gérer les valeurs manquantes
    if pd.isna(value):
        fill_color = 'lightgrey'
        value_display = 'Non disponible'
    else:
        value_percent = value * 100  # Multiplier par 100
        fill_color = colormap(value_percent)
        value_display = f"{value_percent:.2f}%"

    # Récupérer les informations de la commune
    nom = row.get('nom', 'Commune')
    code = row.get('code', 'N/A')
    exprimes = row.get('exprimes', 'N/A')

    popup_text = f"""
    <b>{nom}</b><br>
    Code: {code}<br>
    Vote FN 2017: {value_display}<br>
    Exprimés: {exprimes}
    """

    tooltip_text = f"{nom} - FN: {value_display}"

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#ffffff',
            'weight': 0.5,
            'fillOpacity': 0.7
        },
        highlight_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#000000',
            'weight': 2,
            'fillOpacity': 0.9
        },
        tooltip=tooltip_text,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# Ajouter la légende
colormap.add_to(m)

# Ajouter un titre personnalisé
title_html = '''
<div style="position: fixed;
            bottom: 20px; left: 10px; width: 400px; height: 50px;
            background-color: white; border:2px solid grey; z-index:9999;
            font-size:16px; font-weight: bold; padding: 10px">
Carte du Finistère - Vote Front National en 2017
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

# Sauvegarder et afficher
m.save("carte_finistere_fn2017.html")
print("Carte interactive créée avec coloration selon le vote FN 2017")
m

/var/folders/4z/dn90d96n3v17tkbfc6v_36q40000gn/T/ipykernel_16969/777852524.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [data_map2017.geometry.centroid.y.mean(), data_map2017.geometry.centroid.x.mean()]


Carte interactive créée avec coloration selon le vote FN 2017


In [39]:
color = 'pvoixRN'

center = [data_map2022.geometry.centroid.y.mean(), data_map2022.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=9, tiles="OpenStreetMap")

cmap_mpl = plt.cm.autumn

vmin = data_map2022[color].min() * 100  # Multiplier par 100
vmax = data_map2022[color].max() * 100  # Multiplier par 100

# Extraire les couleurs de la colormap matplotlib 'autumn'
colors_autumn = [mcolors.rgb2hex(cmap_mpl(i)) for i in [0, 0.25, 0.5, 0.75, 1.0]]
colormap = cm.LinearColormap(
    colors=colors_autumn,
    vmin=vmin,
    vmax=vmax,
    caption='Pourcentage de voix RN 2022 (%)'
)

# Ajouter les communes avec coloration selon pvoixFN
for idx, row in data_map2022.iterrows():
    value = row[color]

    # Gérer les valeurs manquantes
    if pd.isna(value):
        fill_color = 'lightgrey'
        value_display = 'Non disponible'
    else:
        value_percent = value * 100  # Multiplier par 100
        fill_color = colormap(value_percent)
        value_display = f"{value_percent:.2f}%"

    # Récupérer les informations de la commune
    nom = row.get('nom', 'Commune')
    code = row.get('code', 'N/A')
    exprimes = row.get('exprimes', 'N/A')

    popup_text = f"""
    <b>{nom}</b><br>
    Code: {code}<br>
    Vote RN 2022: {value_display}<br>
    Exprimés: {exprimes}
    """

    tooltip_text = f"{nom} - RN: {value_display}"

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#ffffff',
            'weight': 0.5,
            'fillOpacity': 0.7
        },
        highlight_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#000000',
            'weight': 2,
            'fillOpacity': 0.9
        },
        tooltip=tooltip_text,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# Ajouter la légende
colormap.add_to(m)

# Ajouter un titre personnalisé
title_html = '''
<div style="position: fixed;
            bottom: 20px; left: 10px; width: 325px; height: 50px;
            background-color: white; border:2px solid grey; z-index:9999;
            font-size:16px; font-weight: bold; padding: 10px">
Carte du Finistère - Vote RN en 2022
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

# Sauvegarder et afficher
m.save("carte_finistere_rn2022.html")
print("Carte interactive créée avec coloration selon le vote RN 2022")
m

/var/folders/4z/dn90d96n3v17tkbfc6v_36q40000gn/T/ipykernel_16969/1222957232.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [data_map2022.geometry.centroid.y.mean(), data_map2022.geometry.centroid.x.mean()]


Carte interactive créée avec coloration selon le vote RN 2022


Création d'une variable d'évolution du vote RN

In [40]:
data_map2022["VarRN"] = (data_map2022["pvoixRN"] - data_map2017["pvoixFN"])
data_map2022.head()

,code,nom,departement,region,commune,plm_x,epci,geometry,dep,nomdep,codecommune,nomcommune,inscrits,votants,exprimes,voixAUG,voixNUP,voixDVG,voixECO,voixREG,voixENS,voixUDI,voixLR,voixDVD,voixREC,voixRN,pvoixAUG,pvoixNUP,pvoixDVG,pvoixECO,pvoixREG,pvoixENS,pvoixUDI,pvoixLR,pvoixDVD,pvoixREC,pvoixRN,pvoixAUGratio,pvoixNUPratio,pvoixDVGratio,pvoixECOratio,pvoixREGratio,pvoixENSratio,pvoixUDIratio,pvoixLRratio,pvoixDVDratio,pvoixRECratio,pvoixRNratio,voteG,voteCG,voteC,voteCD,voteD,voteTG,voteTD,voteGCG,voteDCD,pvoteG,pvoteCG,pvoteC,pvoteCD,pvoteD,pvoteTG,pvoteTD,pvoteGCG,pvoteDCD,pvoteGratio,pvoteCGratio,pvoteCratio,pvoteCDratio,pvoteDratio,pvoteGCGratio,pvoteDCDratio,pvoteTGratio,pvoteTDratio,pervoteG,pervoteCG,pervoteC,pervoteCD,pervoteD,pervoteGCG,pervoteDCD,pervoteTG,pervoteTD,plm_y,plmdoublon,ppar,pparratio,perpar,pblancnul,pblancnulratio,pins,pinsratio,pabs,pblancsnuls,electeurs,VarRN
0,29001,Argol,29,53,None,NaN,200066868,"MULTIPOLYGON (((-4.243 48.249, -4.266 48.23, -...",29,FINISTERE,29001,ARGOL,794,458,444.0,5,123,0,1,29,121,0,52,0,19,94,0.011261,0.277027,0.0,0.002252,0.065315,0.272523,0.0,0.117117,0.000000,0.042793,0.211712,0.960860,1.054070,0.0,0.084040,3.922304,1.051884,0.0,1.106608,0.000000,0.990811,1.100430,133.80000,6.800000,126.80000,57.799999,118.80000,204.0,240.0,140.60001,176.60001,0.301351,0.015315,0.285586,0.130180,0.267568,0.459459,0.540541,0.316667,0.397748,1.084517,0.252696,1.017219,0.917641,1.119951,0.935570,1.044577,0.959505,1.037208,0.636764,0.136368,0.557400,0.582880,0.627832,0.478685,0.565926,0.445837,0.554199,0,0,0.576826,1.176322,0.922028,0.017632,1.682219,1.000000,1.075965,0.423174,0.017632,794.0000,0.063293
1,29002,Arzano,29,53,None,NaN,242900694,"POLYGON ((-3.494 47.902, -3.462 47.903, -3.452...",29,FINISTERE,29002,ARZANO,1071,573,559.0,15,186,0,4,6,182,0,20,11,21,114,0.026834,0.332737,0.0,0.007156,0.010733,0.325581,0.0,0.035778,0.019678,0.037567,0.203936,2.289563,1.266043,0.0,0.267005,0.644563,1.256681,0.0,0.338058,0.601788,0.869816,1.060012,202.20000,5.200000,183.20000,32.200001,136.20000,299.0,260.0,207.39999,168.39999,0.361717,0.009302,0.327728,0.057603,0.243649,0.534884,0.465116,0.371020,0.301252,1.301764,0.153484,1.167326,0.406043,1.019837,1.096152,0.791157,1.117017,0.892481,0.799389,0.086951,0.725744,0.201321,0.549470,0.642695,0.299689,0.692683,0.307345,0,0,0.535014,1.091055,0.752478,0.013072,1.247135,0.816597,0.878631,0.464986,0.013072,1311.5397,0.090217
2,29003,Audierne,29,53,None,NaN,242900629,"POLYGON ((-4.528 48.037, -4.535 48.025, -4.566...",29,FINISTERE,29003,AUDIERNE,3354,1848,1811.0,19,466,0,38,132,594,0,286,0,64,212,0.010491,0.257316,0.0,0.020983,0.072888,0.327996,0.0,0.157924,0.000000,0.035340,0.117062,0.895176,0.979072,0.0,0.782954,4.377052,1.266000,0.0,1.492180,0.000000,0.818242,0.608464,511.39999,64.400002,620.40002,312.399990,302.39999,886.0,925.0,575.79999,614.79999,0.282385,0.035560,0.342573,0.172501,0.166980,0.489232,0.510768,0.317946,0.339481,1.016261,0.586732,1.220202,1.215963,0.698922,0.939349,0.891555,1.021681,0.980078,0.569560,0.369060,0.777460,0.713111,0.262946,0.484311,0.395342,0.545140,0.454935,0,0,0.550984,1.123622,0.828689,0.011032,1.052479,1.000000,1.075965,0.449016,0.011032,3354.0000,0.070039
3,29004,Bannalec,29,53,None,NaN,242900694,"POLYGON ((-3.62 47.927, -3.618 47.903, -3.676 ...",29,FINISTERE,29004,BANNALEC,4575,2343,2243.0,51,757,0,43,61,646,0,117,44,78,446,0.022737,0.337494,0.0,0.019171,0.027196,0.288007,0.0,0.052162,0.019617,0.034775,0.198841,1.940055,1.284144,0.0,0.715337,1.633153,1.111652,0.0,0.492867,0.599910,0.805166,1.033530,820.20001,55.200001,658.20001,173.200000,536.20001,1204.5,1038.5,875.40002,709.40002,0.365671,0.024610,0.293446,0.077218,0.239055,0.537004,0.462996,0.390281,0.316273,1.315993,0.406052,1.045218,0.544310,1.000606,1.153058,0.830605,1.121444,0.888413,0.807417,0.252483,0.585031,0.316642,0.534555,0.699645,0.336736,0.698156,0.301945,0,0,0.512131,1.044390,0.626833,0.021858,2.085373,1.000000,1.075965,0.487869,0.021858,4575.0000,0.094

In [41]:
color = 'VarRN'

center = [data_map2022.geometry.centroid.y.mean(), data_map2022.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=9, tiles="OpenStreetMap")

cmap_mpl = plt.cm.autumn

vmin = data_map2022[color].min() * 100  # Multiplier par 100
vmax = data_map2022[color].max() * 100  # Multiplier par 100

# Extraire les couleurs de la colormap matplotlib 'autumn'
colors_autumn = [mcolors.rgb2hex(cmap_mpl(i)) for i in [0, 0.25, 0.5, 0.75, 1.0]]
colormap = cm.LinearColormap(
    colors=colors_autumn,
    vmin=vmin,
    vmax=vmax,
    caption='Variation du vote RN entre 2017 et 2022 (%)'
)

for idx, row in data_map2022.iterrows():
    value = row[color]

    if pd.isna(value):
        fill_color = 'lightgrey'
        value_display = 'Non disponible'
    else:
        value_percent = value * 100
        fill_color = colormap(value_percent)
        value_display = f"{value_percent:.2f}%"

    nom = row.get('nom', 'Commune')
    code = row.get('code', 'N/A')
    exprimes = row.get('exprimes', 'N/A')

    popup_text = f"""
    <b>{nom}</b><br>
    Code: {code}<br>
    Variation du vote RN 2017 - 2022: {value_display}<br>
    Exprimés: {exprimes}
    """

    tooltip_text = f"{nom} - Variation du vote RN: {value_display}"

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#ffffff',
            'weight': 0.5,
            'fillOpacity': 0.7
        },
        highlight_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#000000',
            'weight': 2,
            'fillOpacity': 0.9
        },
        tooltip=tooltip_text,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

colormap.add_to(m)

title_html = '''
<div style="position: fixed;
            bottom: 20px; left: 10px; width: 500px; height: 50px;
            background-color: white; border:2px solid grey; z-index:9999;
            font-size:16px; font-weight: bold; padding: 10px">
Carte du Finistère - Variation du vote RN entre 2017 et 2022
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

m.save("carte_finistere_rn2022.html")
print("Carte interactive créée avec coloration selon la variation du vote RN entre 2017 et 2022")
m

/var/folders/4z/dn90d96n3v17tkbfc6v_36q40000gn/T/ipykernel_16969/2655498178.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = [data_map2022.geometry.centroid.y.mean(), data_map2022.geometry.centroid.x.mean()]


Carte interactive créée avec coloration selon la variation du vote RN entre 2017 et 2022


In [ ]:
color = 'pvoixRN'

center = [data_map2022.geometry.centroid.y.mean(), data_map2022.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=9, tiles="OpenStreetMap")

cmap_mpl = plt.cm.autumn

vmin = data_map2022[color].min() * 100  # Multiplier par 100
vmax = data_map2022[color].max() * 100  # Multiplier par 100

# Extraire les couleurs de la colormap matplotlib 'autumn'
colors_autumn = [mcolors.rgb2hex(cmap_mpl(i)) for i in [0, 0.25, 0.5, 0.75, 1.0]]
colormap = cm.LinearColormap(
    colors=colors_autumn,
    vmin=vmin,
    vmax=vmax,
    caption='Pourcentage de voix RN 2022 (%)'
)

# Ajouter les communes avec coloration selon pvoixFN
for idx, row in data_map2022.iterrows():
    value = row[color]

    # Gérer les valeurs manquantes
    if pd.isna(value):
        fill_color = 'lightgrey'
        value_display = 'Non disponible'
    else:
        value_percent = value * 100  # Multiplier par 100
        fill_color = colormap(value_percent)
        value_display = f"{value_percent:.2f}%"

    # Récupérer les informations de la commune
    com_name = row.get('com_name', 'Commune')
    com_code = row.get('com_code', 'N/A')
    exprimes = row.get('exprimes', 'N/A')

    popup_text = f"""
    <b>{com_name[0]}</b><br>
    Code: {com_code}<br>
    Vote RN 2022: {value_display}<br>
    Exprimés: {exprimes}
    """

    tooltip_text = f"{com_name[0]} - RN: {value_display}"

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#ffffff',
            'weight': 0.5,
            'fillOpacity': 0.7
        },
        highlight_function=lambda x, color=fill_color: {
            'fillColor': color,
            'color': '#000000',
            'weight': 2,
            'fillOpacity': 0.9
        },
        tooltip=tooltip_text,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# Ajouter la légende
colormap.add_to(m)

# Ajouter un titre personnalisé
title_html = '''
<div style="position: fixed;
            bottom: 20px; left: 10px; width: 325px; height: 50px;
            background-color: white; border:2px solid grey; z-index:9999;
            font-size:16px; font-weight: bold; padding: 10px">
Carte du Finistère - Vote RN en 2022
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

# Sauvegarder et afficher
m.save("carte_finistere_rn2022.html")
print("Carte interactive créée avec coloration selon le vote RN 2022")
m

In [ ]:
data_map2022["VarRN"] = (data_map2022["pvoixRN"] - data_map2017["pvoixFN"])
data_map2022.head()